# P2-2: CoStar pull

Loads all three data layers for one building and reports which come back empty.

1. **CoStar data extract** (`backend/data/costar-hackathon-kit/`): building record, tenants.
2. **Matterport** Model API (GraphQL): floors, rooms, walls, doorways.
3. **Vexcel** aerial imagery (`backend/docs/vexcel/`).

Set ONE of `ADDRESS`, `PROPERTY_ID`, `MATTERPORT_MODEL_ID` below, then run all cells. The last cell prints the summary.
Synthetic/vendor data only; nothing here is saved to git (`backend/data/` is git-ignored).

In [ ]:
# ---- Pick the building (set ONE; leave the others as None) ----
ADDRESS = None              # e.g. "1000 Louisiana St"
PROPERTY_ID = 233899        # CoStar PropertyId (example: Wells Fargo Plaza, Houston)
MATTERPORT_MODEL_ID = None  # e.g. "jUnF84zghQk"

In [ ]:
import csv, json, os
from pathlib import Path

import httpx
from dotenv import load_dotenv

BACKEND = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KIT = BACKEND / "data" / "costar-hackathon-kit"
load_dotenv(BACKEND / ".env")

summary = {}  # layer name -> (status, detail). Status: OK | EMPTY | ERROR | NOT RUN


def record(layer, status, detail):
    summary[layer] = (status, detail)
    print(f"[{status}] {layer}: {detail}")

## Resolve the building

In [ ]:
rows = list(csv.DictReader(open(KIT / "commercial-properties.csv", encoding="utf-8")))

if PROPERTY_ID is not None:
    hits = [r for r in rows if r["PropertyId"] == str(PROPERTY_ID)]
elif MATTERPORT_MODEL_ID:
    hits = [r for r in rows if r["MatterportModelId"] == MATTERPORT_MODEL_ID]
elif ADDRESS:
    hits = [r for r in rows if ADDRESS.lower() in r["Address"].lower()]
else:
    raise SystemExit("Set ADDRESS, PROPERTY_ID or MATTERPORT_MODEL_ID above.")

assert hits, "No match in commercial-properties.csv (it only lists the 3,339 Matterport-listed properties)."
property_ids = sorted({r["PropertyId"] for r in hits})
if len(property_ids) > 1:
    print("Several properties match, using the first:", property_ids)
PID = property_ids[0]
first = next(r for r in hits if r["PropertyId"] == PID)
MODEL_IDS = [r["MatterportModelId"] for r in rows if r["PropertyId"] == PID and r["MatterportModelId"]]
print(f"Property {PID}: {first['Address']}, {first['City']}, {first['State']}")
print(f"Matterport models in the kit CSV: {len(MODEL_IDS)} -> {MODEL_IDS}")

## Layer 1: CoStar building data

In [ ]:
def load_property(pid):
    prefix = f'{{"id":{pid},'
    with open(KIT / "data" / "commercial-property.jsonl", encoding="utf-8") as f:
        for line in f:
            if line.startswith(prefix):
                return json.loads(line)


def find_key(obj, key):
    # Elevators live inside the type block (propertyOffice.structural.elevators, ...), so search for them.
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            found = find_key(v, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for v in obj:
            found = find_key(v, key)
            if found is not None:
                return found


enums = json.load(open(KIT / "schemas" / "enums.json", encoding="utf-8"))
decode = lambda enum, v: enums[enum].get(str(v), v)

prop = load_property(PID)
assert prop, f"Property {PID} not found in commercial-property.jsonl"
addr = prop["primaryAddress"]

elevators = [
    {"type": decode("ElevatorType", e.get("elevatorType")), "count": e.get("numElevators"), "known": e.get("elevatorState") == 1}
    for e in (find_key(prop, "elevators") or [])
]
parking = [{"type": decode("ParkingType", p.get("parkingType")), "spaces": p.get("totalSpaces"), "monthly_fee": p.get("monthlyParkingFee")} for p in prop.get("parking") or []]
mp_attachments = [a["videoKey"] for a in prop.get("attachments") or [] if a.get("videoExternalReferenceType") in (36, 234)]

building = {
    "id": prop["id"],
    "name": prop.get("buildingName"),
    "address": f"{addr.get('streetNumber')} {addr.get('streetName')} {addr.get('streetType')}, {addr.get('city')}, {addr.get('state')}",
    "lat_lon": (addr.get("latitude"), addr.get("longitude")),
    "type": decode("PropertyType", prop.get("propertyType")),
    "stories_above_grade": prop.get("numStoriesAboveGrade"),
    "floors_listed": len(prop.get("floors") or []),
    "typical_floor_sqft": prop.get("grossTypicalFloorSquareFeet"),
    "elevators": elevators,
    "parking": parking,
    "matterport_attachments": mp_attachments,
}
print(json.dumps(building, indent=2, default=str))

empty = [k for k in ("name", "stories_above_grade", "typical_floor_sqft") if not building[k]]
empty += [k for k in ("floors_listed", "elevators", "parking") if not building[k]]
if not all(building["lat_lon"]):
    empty.append("lat_lon")

In [ ]:
# Tenants: tenancies on this property, joined to company names.
tenancies = []
needle = f'"propertyId":{PID},'
with open(KIT / "data" / "location-occupancy.jsonl", encoding="utf-8") as f:
    for line in f:
        if needle in line:
            r = json.loads(line)
            if r["propertyId"] == int(PID):
                tenancies.append(r)

company_ids = {t["tenantLocationId"] for t in tenancies if t.get("tenantLocationId")}
companies = {}
with open(KIT / "data" / "company-detail.jsonl", encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        if c["id"] in company_ids:
            companies[c["id"]] = (c.get("name"), (c.get("suite") or "").strip())

print(f"{len(tenancies)} tenancies, {len(companies)} named companies. Sample:")
for name, suite in list(companies.values())[:8]:
    print("  ", name, "| suite", suite or "?")
if not tenancies:
    empty.append("tenants")

if empty:
    record("1 CoStar data", "EMPTY" if len(empty) > 3 else "OK", f"building record found; empty fields: {empty}")
else:
    record("1 CoStar data", "OK", f"{building['name']}, {building['stories_above_grade']} floors, {len(tenancies)} tenancies")

## Layer 2: Matterport (Model API, GraphQL)

Endpoint `https://api.matterport.com/api/models/graph`, HTTP Basic auth: API key (token ID) as username, API token (secret) as password
(`MATTERPORT_API_KEY`, `MATTERPORT_API_TOKEN` in `backend/.env`).
Floors and rooms come from the basic query; walls and doorways ("vector data") are a Matterport Enterprise feature, so they are queried separately and may come back empty.

In [ ]:
MP_URL = "https://api.matterport.com/api/models/graph"
MP_ID = os.getenv("MATTERPORT_API_KEY")
MP_SECRET = os.getenv("MATTERPORT_API_TOKEN")

BASIC = '''
query basic($id: ID!) {
  model(id: $id) {
    id name created modified visibility
    floors { id sequence label }
    rooms { id label classifications { label } }
  }
}'''

VECTOR = '''
query vector($id: ID!) {
  model(id: $id) {
    rooms {
      id label
      boundary { edges { id type thickness
        openings { id type width height lowerElevation relativeCenter }
        vertices { id position { x y } } } }
    }
  }
}'''


def mp(query, model_id):
    r = httpx.post(MP_URL, json={"query": query, "variables": {"id": model_id}}, auth=(MP_ID, MP_SECRET), timeout=60)
    if r.status_code in (401, 403):
        raise RuntimeError(f"HTTP {r.status_code}: check the token id/secret and that they can see this model")
    r.raise_for_status()
    return r.json()


mp_results = {}
if not (MP_ID and MP_SECRET):
    record("2 Matterport", "NOT RUN", "MATTERPORT_API_KEY / MATTERPORT_API_TOKEN are not set in backend/.env")
elif not MODEL_IDS:
    record("2 Matterport", "EMPTY", "this property has no Matterport model in the kit")
else:
    for mid in MODEL_IDS[:5]:
        entry = {"model": mid}
        try:
            body = mp(BASIC, mid)
            if body.get("errors"):
                entry["error"] = body["errors"][0].get("message")
            else:
                m = (body.get("data") or {}).get("model") or {}
                entry.update(name=m.get("name"), floors=len(m.get("floors") or []), rooms=len(m.get("rooms") or []))
            vbody = mp(VECTOR, mid)
            if vbody.get("errors"):
                entry["vector_error"] = vbody["errors"][0].get("message")
            else:
                rooms = ((vbody.get("data") or {}).get("model") or {}).get("rooms") or []
                edges = [e for r in rooms for e in ((r.get("boundary") or {}).get("edges") or [])]
                entry.update(wall_edges=len(edges), doorways=sum(len(e.get("openings") or []) for e in edges))
        except Exception as e:
            entry["error"] = str(e)
        mp_results[mid] = entry
        print(entry)

    usable = [e for e in mp_results.values() if e.get("floors") and e.get("rooms")]
    with_geometry = [e for e in usable if e.get("wall_edges")]
    if with_geometry:
        record("2 Matterport", "OK", f"{len(with_geometry)}/{len(mp_results)} models return floors, rooms, walls and doorways")
    elif usable:
        record("2 Matterport", "OK", f"{len(usable)} models return floors and rooms, but no wall/doorway geometry (vector data is Enterprise-only?)")
    elif all("error" in e for e in mp_results.values()):
        record("2 Matterport", "ERROR", next(iter(mp_results.values()))["error"])
    else:
        record("2 Matterport", "EMPTY", "queries worked but returned no floors or rooms")

## Layer 3: Vexcel aerial imagery

Needs the tutorial PDF and `vexcel_quickstart.py` in `backend/docs/vexcel/`. This cell does not guess the API: it only runs once those files exist.

In [ ]:
VEXCEL_DIR = BACKEND / "docs" / "vexcel"
quickstart = VEXCEL_DIR / "vexcel_quickstart.py"

if not quickstart.exists():
    record("3 Vexcel", "NOT RUN", f"{quickstart} not found, so the Vexcel API is unknown. Add the tutorial PDF and quickstart there.")
else:
    print(quickstart.read_text(encoding="utf-8")[:4000])
    record("3 Vexcel", "NOT RUN", "quickstart found but the fetch is not written yet: read it above, then add the call here")

## Summary

In [ ]:
print(f"Building: {building['name']} ({building['address']})\n")
for layer, (status, detail) in sorted(summary.items()):
    print(f"{status:8} {layer}: {detail}")
empties = [k for k, (s, _) in summary.items() if s != "OK"]
print("\nLayers not OK:", empties or "none")